# Question13  
This question should be answered using the Weekly data set, which is part of the ISLP package.  

In [39]:
# packages
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)
from ISLP.models import contrast
from ISLP import confusion_table
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
# setting
pd.set_option('display.float_format', '{:.3f}'.format)

In [40]:
df = load_data('Weekly')
df

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.155,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.149,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.160,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.162,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.154,1.178,Up
...,...,...,...,...,...,...,...,...,...
1084,2010,-0.861,0.043,-2.173,3.599,0.015,3.205,2.969,Up
1085,2010,2.969,-0.861,0.043,-2.173,3.599,4.243,1.281,Up
1086,2010,1.281,2.969,-0.861,0.043,-2.173,4.835,0.283,Up
1087,2010,0.283,1.281,2.969,-0.861,0.043,4.454,1.034,Up


**Column Description**  
This dataset contains the weekly trading volume and return rate of the S&P 500 from 1990 to 2010.

`Lag1` – `Lag5`: Weekly return rates from the previous 1 to 5 weeks. These are predictor variables.  
`Volume`: Trading volume of the current week. This is a predictor variable.  
`Today`: Weekly return rate of the current week. This is a predictor variable.  
`Direction`: Market direction for the current week, indicating whether it went "Up" or "Down". This is the target variable.


in Question (d), we've fitted the logistic regression model using a training data period from 1990 to 2008, with Lag2 as the only predictor and computed the confusion matrix and the overall fraction of correct predictions for the held out data (that is, the data from 2009 and 2010).

In [41]:
train = (df.Year <= 2008)
df_train = df.loc[train]
df_test = df.loc[~train]
df_test.shape

df1 = ['Lag1', 'Lag3', 'Lag4', 'Lag5', 'Volume', 'Today', 'Direction', 'Year']
train_cols = df_train.columns.drop(df1)

design = MS(train_cols) # 根據選定的變數 train_cols 來建立設計矩陣（design matrix），也就是機器學習裡的 X。

X_train = design.fit_transform(df_train)
y_train = df_train.Direction == 'Up'

X_test = design.transform(df_test)
y_test = df_test.Direction == 'Up'

glm = sm.GLM(y_train, X_train, family=sm.families.Binomial())
results = glm.fit()
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:              Direction   No. Observations:                  985
Model:                            GLM   Df Residuals:                      983
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -675.27
Date:                Fri, 07 Nov 2025   Deviance:                       1350.5
Time:                        13:53:58   Pearson chi2:                     985.
No. Iterations:                     4   Pseudo R-squ. (CS):           0.004221
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.2033      0.064      3.162      0.0

In [44]:
probs_test = results.predict(X_test)
labels_test = np.array(['Down'] * len(probs_test))
labels_test[probs_test > 0.5] = 'Up'
confusion_table(labels_test, df_test.Direction, labels=['Up','Down'])

Truth,Up,Down
Predicted,,
Up,56,34
Down,5,9


### (e)  
Repeat (d) using LDA.

In [45]:
lda = LDA(store_covariance=True)

X_train, X_test = [M.drop(columns=['intercept'])
                   for M in [X_train, X_test]]
lda.fit(X_train, y_train)

,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None
,store_covariance,True
,tol,0.0001
,covariance_estimator,None


In [46]:
lda.means_

array([[-0.03568254],
       [ 0.26036581]])

In [47]:
lda.classes_

array([False,  True])

In [48]:
lda.priors_
# which means pi False is 0.45 and pi True is 0.55

array([0.44771574, 0.55228426])

In [53]:
lda.scalings_
# 在分類時，原始每個特徵要被加多少權重，才能形成最能分開各類別的新軸

array([[0.44141622]])

In [55]:
lda_pred = lda.predict(X_test)
confusion_table(lda_pred, y_test)

Truth,False,True
Predicted,,
False,9,5
True,34,56


**Notes**  
Accuracy = 0.63 = (56+9)/(56+9+34+5)  
TPR = 0.91 = 56/(56+5)  
FPR = 0.79 = 34/(34+9)  

### (f)  
Repeat (d) using QDA.

In [56]:
qda = QDA(store_covariance=True)
qda.fit(X_train, y_train)

,priors,None
,reg_param,0.0
,store_covariance,True
,tol,0.0001


In [59]:
qda_pred = qda.predict(X_test)
confusion_table(qda_pred, y_test)

Truth,False,True
Predicted,,
False,0,0
True,43,61


**Notes**  
Accuracy = 0.59 = (61+0)/(61+0+43+0)  
TPR = 1.00 = 61/(61+0)  
FPR = 1.00 = 43/(43+0)  

### (g)  
Repeat (d) using KNN with K =1.

In [60]:
Knn1 = KNeighborsClassifier(n_neighbors=1)
Knn1.fit(X_train, y_train)
Knn1_pred = Knn1.predict(X_test)
confusion_table(Knn1_pred, y_test)

Truth,False,True
Predicted,,
False,22,31
True,21,30


**Notes**  
Accuracy: 0.5 = (30+22)/(30+22+21+31)  
TPR: 0.49 = 30/(30+31)  
FPR: 0.48 = 21/(21+22)

### (h)  
Repeat (d) using naive Bayes.

In [61]:
NB = GaussianNB()
NB.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


In [72]:
nb_labels = NB.predict(X_test)
confusion_table(nb_labels, y_test)

Truth,False,True
Predicted,,
False,0,0
True,43,61


**Notes**  
Accuracy: 0.58 = (61+0)/(61+0+43+0)  
TPR: 1.00 = 61/(61+0)  
FPR: 1.00 = 43/(43+0)

### (i)  
Which of these methods appears to provide the best results on this data?

### (j)  
Experiment with different combinations of predictors, including possible transformations and interactions, for each of the methods. Report the variables, method, and associated confusion matrix that appears to provide the best results on the held out data. Note that you should also experiment with values for K in the KNN classifier.